# Benchmark: Object-Mass Feasible 5% (Lightweight)

Легковесный второй ноутбук для датасета `p005` из
`demo/data/object_mass_feasible_task_sweep_5pct_fullfleet/`.

Что оставили:
- быстрый benchmark только по 3 алгоритмам: `real_gap_vrp_alns`, `real_milp`, `real_genetic`;
- основная итоговая таблица.

Что убрали как долгие части:
- per-agent карты;
- большие детальные распечатки по каждому алгоритму.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import time

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option("display.max_colwidth", 140)
print("REPO_ROOT:", REPO_ROOT)


REPO_ROOT: /Users/igoreshka/Desktop/Optimization-of-flows


In [2]:
from flowopt.demo import prepare_context_from_dataset_path

DATASET_PATH = (
    REPO_ROOT
    / "demo"
    / "data"
    / "object_mass_feasible_task_sweep_5pct_fullfleet"
    / "dataset_real_spb_day_object_mass_feasible_tasks_p005.json"
)

SUMMARY_PATH = (
    REPO_ROOT
    / "demo"
    / "data"
    / "object_mass_feasible_task_sweep_5pct_fullfleet"
    / "summary_real_spb_day_object_mass_feasible_tasks_p005.json"
)

context = prepare_context_from_dataset_path(
    dataset_path=DATASET_PATH,
    summary_path=SUMMARY_PATH,
    output_subdir="object_mass_feasible_5pct_demo",
)

print("DATASET_PATH:", context.dataset_path)
print("SUMMARY_PATH:", context.summary_path)
print("OUTPUT_DIR  :", context.output_dir)
pd.DataFrame([context.counts])


DATASET_PATH: /Users/igoreshka/Desktop/Optimization-of-flows/demo/data/object_mass_feasible_task_sweep_5pct_fullfleet/dataset_real_spb_day_object_mass_feasible_tasks_p005.json
SUMMARY_PATH: /Users/igoreshka/Desktop/Optimization-of-flows/demo/data/object_mass_feasible_task_sweep_5pct_fullfleet/summary_real_spb_day_object_mass_feasible_tasks_p005.json
OUTPUT_DIR  : /Users/igoreshka/Desktop/Optimization-of-flows/demo/local/object_mass_feasible_5pct_demo/dataset_real_spb_day_object_mass_feasible_tasks_p005


,nodes,edges,agents,tasks,objects,sources
0,46730,127100,626,1071,9,904


In [5]:
from flowopt.pipeline_runtime import execute_solver
from flowopt.demo import benchmark_main_table

SHOW_ALGO_PROGRESS = True
SHOW_SOLVER_DETAILS = True

ALGORITHMS = [
    # (
    #     "real_gap_vrp_alns",
    #     {
    #         "step1_method": "dataset",
    #         "gap_iter": 12,
    #         "use_repair": True,
    #         "alns_iterations": 12,
    #         "alns_removal_q": 4,
    #         "alns_seed": 42,
    #         "start_temperature": 70.0,
    #         "end_temperature": 2.0,
    #         "temperature_step": 0.98,
    #     },
    # ),
    (
        "real_milp",
        {
            "time_limit_sec": 60,
            "unassigned_penalty": 1e7,
        },
    ),
    # (
    #     "real_genetic",
    #     {
    #         "population_size": 18,
    #         "generations": 18,
    #         "elite_size": 3,
    #         "seed": 42,
    #         "generation_scale": 1.0,
    #         "min_generations": 8,
    #         "max_runtime_sec": 90.0,
    #     },
    # ),
]

def run_algo(algorithm: str, params: dict):
    t0 = time.perf_counter()
    if SHOW_ALGO_PROGRESS:
        print(f"[start] {algorithm}")
    execution = execute_solver(
        algorithm=algorithm,
        dataset_path=context.dataset_path,
        solver_kwargs=params,
        show_progress=SHOW_SOLVER_DETAILS,
        verbose=False,
    )
    elapsed = time.perf_counter() - t0
    if SHOW_ALGO_PROGRESS:
        print(f"[done]  {algorithm}: {elapsed:.1f}s")
    return execution.metrics.as_dict()

rows = [run_algo(algo, params) for algo, params in ALGORITHMS]
benchmark_df = pd.DataFrame(rows).sort_values(
    by=["all_checks_ok", "transport_work_ton_km", "total_km", "runtime_sec"],
    ascending=[False, True, True, True],
    na_position="last",
).reset_index(drop=True)

benchmark_main_table(benchmark_df)


[start] real_milp
[real_milp] dataset profile: generic_real
[real_milp] constraints: edge_vt=True, agent_volume=False, object_caps_m=True, object_caps_v=False
[real_milp] build nx graph
[real_milp] solver start
[real_milp] prepare problem: tasks=1071, agents=626
[real_milp] compatible pairs: 107/1071 tasks processed, pairs=41916
[real_milp] compatible pairs: 214/1071 tasks processed, pairs=84086
[real_milp] compatible pairs: 321/1071 tasks processed, pairs=125681
[real_milp] compatible pairs: 428/1071 tasks processed, pairs=168091
[real_milp] compatible pairs: 535/1071 tasks processed, pairs=210136
[real_milp] compatible pairs: 642/1071 tasks processed, pairs=252839
[real_milp] compatible pairs: 749/1071 tasks processed, pairs=296118
[real_milp] compatible pairs: 856/1071 tasks processed, pairs=338500
[real_milp] compatible pairs: 963/1071 tasks processed, pairs=380782
[real_milp] compatible pairs: 1070/1071 tasks processed, pairs=423225
[real_milp] compatible pairs: 1071/1071 tasks pr

,algorithm,feasible,all_checks_ok,assigned_routes,unassigned_tasks,active_agents,transport_work_ton_km,total_km,deadhead_km,deadhead_share_pct,total_hours,runtime_sec
0,real_milp,False,False,983,88,400,12441.984,55377.024,21680.165,39.15,2498.913,249.727


In [6]:
# Опционально: посмотреть полный DataFrame
SHOW_FULL_TABLE = False
if SHOW_FULL_TABLE:
    benchmark_df
else:
    print("Set SHOW_FULL_TABLE=True to display full benchmark_df")


Set SHOW_FULL_TABLE=True to display full benchmark_df


In [ ]:
# Опционально: отрисовать карту только для одного алгоритма (может быть небыстро)
RENDER_ONE_MAP = False
MAP_ALGO = "real_milp"

if RENDER_ONE_MAP:
    from flowopt.demo import render_solution_map_for_algorithm
    import matplotlib.image as mpimg
    import matplotlib.pyplot as plt

    render_info = render_solution_map_for_algorithm(
        context,
        algorithm=MAP_ALGO,
        show_solver_progress=False,
    )
    print(render_info)

    img = mpimg.imread(render_info["map_path"])
    plt.figure(figsize=(14, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Schedule Map: {MAP_ALGO}")
    plt.show()
else:
    print("Map rendering skipped (set RENDER_ONE_MAP=True to enable).")


KeyboardInterrupt: 